In [283]:
import os
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video
import imageio
from statistics import mean
import robomimic.utils.file_utils as FileUtils
from PIL import Image, ImageDraw, ImageFont
import zarr
from diffusion_policy.common.replay_buffer import ReplayBuffer
from filelock import FileLock
from diffusion_policy.codecs.imagecodecs_numcodecs import register_codecs, Jpeg2k
import pdb
from tqdm import tqdm
import xml.etree.ElementTree as ET


In [7]:
og_redcube_data = h5py.File('/proj/vondrick3/sruthi/robots/diffusion_policy/data/robomimic/datasets/lift/ph/robomimic/datasets/lift/ph/image_abs.hdf5', 'r')
print(og_redcube_data['data']['demo_9'].keys())

<KeysViewHDF5 ['actions', 'dones', 'next_obs', 'obs', 'rewards', 'states']>


In [241]:
trial_hammer_basepath = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_14_23_34_15/'
obsdict_agentview = np.load(trial_hammer_basepath + 'obsdict_agentview.npy', 'r')
obsdict_eyeinhand = np.load(trial_hammer_basepath + 'obsdict_eyeinhand.npy', 'r')
obsdict_robot0s = np.load(trial_hammer_basepath + 'obsdict_robot0s.npy', 'r')
rewards = np.load(trial_hammer_basepath + 'rewards.npy', 'r')
states = np.load(trial_hammer_basepath + 'startstates.npy', 'r')
actions = np.load(trial_hammer_basepath + 'actions.npy', 'r')

In [107]:
print('hi', trial_hammer_basepath)
obsdict_agentview = (obsdict_agentview.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_agentview', obsdict_agentview.shape)

obsdict_eyeinhand = (obsdict_eyeinhand.transpose(0,2,1,4,5,3).reshape(13*8,1008,84,84,3)* 255.0).astype(np.uint8)[:-4]
print('obsdict_eyeinhand', obsdict_eyeinhand.shape)

obsdict_robot0s = obsdict_robot0s.transpose(0,2,1,3).reshape(13*8,1008,9)[:-4]
print('obsdict_robot0s', obsdict_robot0s.shape)

actions = np.load(trial_hammer_basepath + 'actions.npy', 'r')
print(actions.shape)
actions = actions[:,:,:8,:].transpose(0,2,1,3).reshape(13*8,1008,7)[:-4]
print('actions', actions.shape)

rewards = rewards.transpose(1,0)
print('rewards', rewards.shape)

states = np.repeat(np.array(states)[np.newaxis, :, :], obsdict_agentview.shape[0], axis=0)
print('states', states.shape)

hi /proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_14_23_34_15/
obsdict_agentview (100, 1008, 84, 84, 3)
obsdict_eyeinhand (100, 1008, 84, 84, 3)
obsdict_robot0s (100, 1008, 9)
(13, 1008, 8, 7)
actions (100, 1008, 7)
rewards (100, 1008)
states (100, 1008, 32)


expected:

hi
obsdict_agentview (100, 1008, 84, 84, 3)
obsdict_eyeinhand (100, 1008, 84, 84, 3)
obsdict_robot0s (100, 1008, 9)
(13, 1008, 8, 7)
actions (100, 1008, 7)
rewards (100, 1008)
states (100, 1008, 32)

In [108]:
trial_hammer_basepath

'/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_14_23_34_15/'

In [102]:
''' CREATE 1 DATASET '''
new_data = {'data':{}}

count = 0
'''
#for high quality dataset
for datapt in range(0,rewards.shape[1]):
    success = int(np.max(rewards[:,datapt]))
    minsuccess = min(np.argwhere(rewards[:,datapt]==1))[0] if success else -1

    if minsuccess>0 and rewards[:,datapt][minsuccess:minsuccess+10].sum()==10:
        stop_demo = minsuccess+16
        new_data['data'][f'demo_{count}'] = {
            'rewards': rewards[:stop_demo,datapt],
            'success': [success],
            'og_pt': [datapt],
            'states': states[:stop_demo,datapt,:],
            'actions': actions[:stop_demo, datapt],
            'obs': {
                'agentview_image': obsdict_agentview[:stop_demo, datapt],
                'robot0_eye_in_hand_image': obsdict_eyeinhand[:stop_demo, datapt],
                'robot0_eef_pos': obsdict_robot0s[:stop_demo, datapt, :3],
                'robot0_eef_quat': obsdict_robot0s[:stop_demo, datapt, 3:7],
                'robot0_gripper_qpos': obsdict_robot0s[:stop_demo, datapt, 7:],
            }
        }
        count+=1 
    else:
        stop_demo = -1
'''

for datapt in range(0,rewards.shape[1]):
    success = np.max(rewards[:,datapt])
    if success>0:
        stop_demo = min(np.argwhere(rewards[:,datapt]==1))[0] + 16
        new_data['data'][f'demo_{count}'] = {
            'rewards': rewards[:stop_demo,datapt],
            'success': [success],
            'og_pt': [datapt],
            'states': states[:stop_demo,datapt,:],
            'actions': actions[:stop_demo, datapt],
            'obs': {
                'agentview_image': obsdict_agentview[:stop_demo, datapt],
                'robot0_eye_in_hand_image': obsdict_eyeinhand[:stop_demo, datapt],
                'robot0_eef_pos': obsdict_robot0s[:stop_demo, datapt, :3],
                'robot0_eef_quat': obsdict_robot0s[:stop_demo, datapt, 3:7],
                'robot0_gripper_qpos': obsdict_robot0s[:stop_demo, datapt, 7:],
            }
        }
        count+=1 
    else:
        stop_demo = -1   
    
# Open HDF5 file and write in the data_dict structure and info
savepath = trial_hammer_basepath+'data_successful_only.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')


datagrp.attrs['env_args'] = og_redcube_data['data'].attrs['env_args']
datagrp.attrs['total'] = len(new_data['data'])


for demo in new_data['data']:
    demogrp = datagrp.create_group(demo)
    
    actionsdset = demogrp.create_dataset('actions', data = new_data['data'][demo]['actions'])
    rewardsdset = demogrp.create_dataset('rewards', data = new_data['data'][demo]['rewards'])
    successdset = demogrp.create_dataset('success', data = new_data['data'][demo]['success'])
    statesdset = demogrp.create_dataset('states', data = new_data['data'][demo]['states'])
    statesdset = demogrp.create_dataset('og_pt', data = new_data['data'][demo]['og_pt'])

    obsgrp = demogrp.create_group('obs') 
    for grp_name in new_data['data'][demo]['obs']:
        dset = obsgrp.create_dataset(grp_name, data = new_data['data'][demo]['obs'][grp_name])
print('demo done', demo)
f.close()


demo done demo_612


In [48]:
print(trial_hammer_basepath+'data_all.hdf5')
print(h5py.File(trial_hammer_basepath+'data_alll.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_unsuccessful_only.hdf5')['data'])
print(h5py.File(trial_hammer_basepath+'data_successful_only.hdf5')['data'])

/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_6_11_37_51/data_all.hdf5


FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = '/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2024.09.03/21.23.37_train_diffusion_unet_hybrid_15.00.33_check/checkpoints/epoch=0150-test_mean_score=0.940/unguided/alift_4wredcube_11_6_11_37_51/data_alll.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [49]:
'''TEST THE NEW DATASET'''

data = h5py.File(trial_hammer_basepath + 'data_all.hdf5', 'r')
print(data['data'])
video_path = trial_hammer_basepath+'temp.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 50
demo = f'demo_{idx}'
print('shape', data['data'][demo]['obs']['agentview_image'].shape)
print('success', data['data'][demo]['success'][:])
print('ogpt', data['data'][demo]['og_pt'][0])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (84, 84) to (96, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


<HDF5 group "/data" (1008 members)>
shape (43, 84, 84, 3)
success [1.]
ogpt 50


[swscaler @ 0x686b280] Warning: data is not aligned! This can lead to a speed loss


# Testing other datasets

In [ ]:
for root, _, files in os.walk('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage'):
    for file in files:
        if file.endswith(".hdf5"):
            try:
                data=h5py.File(os.path.join(root, file), 'r')
            except:
                print('could not find', root,file)
            temp = json.loads(data['data']['demo_1'].attrs['ep_meta'])
            envname=json.loads(data['data'].attrs['env_args'])['env_name']
            if 'PnP' in envname:
                print(  envname, ' ex: ', temp['lang'])
                for oc in temp['object_cfgs']:
                    if 'graspable' in oc and oc['graspable']:
                        print(oc['info']['cat'],oc['info']['mjcf_path'].split('/')[-3:-1],'g',oc.get('obj_groups',''),'e',oc.get('exclude_obj_groups',''),'w',oc.get('washable',''),'m',oc.get('microwavable',''),'c',oc.get('cookable',''),'f',oc.get('freezable',''),'ms',oc.get('max_size',''),'os',oc.get('object_scale',''))
                        print(temp['lang'].split(' ')[2]==oc['info']['cat'])
            # for i in data['data']:
            #     print(i)
            #     if 'ep_meta' in data['data']['demo_1'].attrs and 'lang' in json.loads(data['data']['demo_1'].attrs['ep_meta']):
            #         print(json.loads(data['data'][i].attrs["ep_meta"])['lang'])

In [ ]:
path='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams_new_images.hdf5'
data=h5py.File(path,'r')
demo_keys = sorted(data['data'], key=lambda x: int(x.split('_')[1]))
ep_lens = []
langset = []
for demo in demo_keys:
    ep_lens.append(data['data'][demo]['actions'].shape[0])
    max_ep=max(max_ep,data['data'][demo]['actions'].shape[0])
    langset.append(demo+': '+json.loads(data['data'][demo].attrs['ep_meta'])['lang'].split('from the pan and place it on the plate')[0])
print('len',len(ep_lens))
print('max',max(ep_lens))
print('avg',mean(ep_lens))
print('langset',*langset, sep="\n")

In [657]:
json.loads(data['data'].attrs['env_args'])

{'env_name': 'PnPStoveToCounter',
 'env_version': '1.5.0',
 'type': 1,
 'env_kwargs': {'robots': 'PandaMobile',
  'controller_configs': {'type': 'OSC_POSE',
   'input_max': 1,
   'input_min': -1,
   'output_max': [0.05, 0.05, 0.05, 0.5, 0.5, 0.5],
   'output_min': [-0.05, -0.05, -0.05, -0.5, -0.5, -0.5],
   'kp': 150,
   'damping_ratio': 1,
   'impedance_mode': 'fixed',
   'kp_limits': [0, 300],
   'damping_ratio_limits': [0, 10],
   'position_limits': None,
   'orientation_limits': None,
   'uncouple_pos_ori': True,
   'control_delta': True,
   'interpolation': None,
   'ramp_ratio': 0.2},
  'layout_ids': -1,
  'style_ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 11],
  'translucent_robot': False,
  'obj_instance_split': 'A',
  'generative_textures': '100p',
  'randomize_cameras': True,
  'camera_heights': 128,
  'camera_widths': 128,
  'has_renderer': False,
  'has_offscreen_renderer': True,
  'ignore_done': True,
  'use_object_obs': True,
  'use_camera_obs': True,
  'camera_depths': False,
  'r

In [646]:
json.loads(data['data']['demo_0'].attrs['ep_meta'])['cam_configs']

{'robot0_agentview_center': {'pos': [-0.5096122227032206,
   -0.05667804521586239,
   1.155836420815279],
  'quat': [-0.6376155856169933,
   -0.3337631286815403,
   0.313302753849731,
   0.6195885113050299],
  'parent_body': 'mobilebase0_support'},
 'robot0_agentview_left': {'pos': [-0.524529335973194,
   0.2523581996318248,
   1.0920631916930328],
  'quat': [-0.5706848525335162,
   -0.2769670975859552,
   0.38731956472371887,
   0.6690228551595528],
  'camera_attribs': {'fovy': '60'},
  'parent_body': 'mobilebase0_support'},
 'robot0_agentview_right': {'pos': [-0.5653366692201566,
   -0.3867171871337491,
   1.0872504139710986],
  'quat': [-0.6971511494273813,
   -0.37934091249691465,
   0.3289211124189629,
   0.5117535039090221],
  'camera_attribs': {'fovy': '60'},
  'parent_body': 'mobilebase0_support'},
 'robot0_frontview': {'pos': [-0.5, 0, 0.95],
  'quat': [0.6088936924934387,
   0.3814677894115448,
   -0.3673907518386841,
   -0.5905545353889465],
  'camera_attribs': {'fovy': '60'

In [606]:
data['data']['demo_100'].attrs.keys()

<KeysViewHDF5 ['dataset_path', 'ep_meta', 'model_file', 'num_samples', 'og_demo_id']>

In [615]:
data['data']['demo_200'].attrs['num_samples']

332

In [ ]:
basepath='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/combined/PnPXToCounter/demo_gentex_im128_randcams_new_images.hdf5'
data=h5py.File(f'{basepath}', 'r')

for side in ['agentview_left','agentview_right','eye_in_hand']:
    if not os.path.exists(basepath + f'/demos/{side}'):
        os.makedirs(basepath + f'/demos/{side}')
    vidcount=0
    for demo in data['data']:
        if vidcount < 11:
            video_path = f'{basepath}/demos/{side}/{demo}.mp4'
            video_writer = imageio.get_writer(video_path, fps=20)
            if 'success' in data['data'][demo]:
                print('success',data['data'][demo]['success'][0])
            idx=0
            for b in  data['data'][demo]['obs']['robot0_'+side+'_image']:
                img = Image.fromarray((b).astype(np.uint8))
                d = ImageDraw.Draw(img)
                d.text( (2,2), str(idx), fill=255)
                b = np.asarray(img)
                video_writer.append_data(b)
                idx+=1
            video_writer.close()
            Video(video_path, embed=True)
            vidcount+=1

data.close()

# Combine Datasets

# These are the tasks we are combingin into 1 dataset:


In [636]:
paths_x_to_counter = [
        '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images.hdf5',
         '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams_new_images.hdf5',
         '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPMicrowaveToCounter/2024-04-26/demo_gentex_im128_randcams_new_images.hdf5',
         '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCabToCounter/2024-04-24/demo_gentex_im128_randcams_new_images.hdf5'
]
paths_counter_to_x = [
    '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToMicrowave/2024-04-27/demo_gentex_im128_randcams_new_images.hdf5',
    '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToSink/2024-04-25/demo_gentex_im128_randcams_new_images.hdf5',
    '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToStove/2024-04-26/demo_gentex_im128_randcams_new_images.hdf5',
    '/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToCab/2024-04-24/demo_gentex_im128_randcams_new_images.hdf5'
]
paths=paths_x_to_counter+paths_counter_to_x
print(*paths, sep="\n")

/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPStoveToCounter/2024-05-01/demo_gentex_im128_randcams_new_images.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPMicrowaveToCounter/2024-04-26/demo_gentex_im128_randcams_new_images.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCabToCounter/2024-04-24/demo_gentex_im128_randcams_new_images.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToMicrowave/2024-04-27/demo_gentex_im128_randcams_new_images.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToSink/2024-04-25/demo_gentex_im128_randcams_new_images.hdf5
/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPCounterToStov

In [631]:
f.close()

In [637]:
''' CREATE 1 DATASET '''
# Open HDF5 file and write in the data_dict structure and info
base_path = 'test'
savepath = base_path+'/test.hdf5'
f = h5py.File(savepath, 'w')
datagrp = f.create_group('data')
datagrp.attrs['datasets_included'] = ', '.join(paths)
datagrp.attrs['env_args'] = testdata['data'].attrs['env_args']
total=0
for i in paths:
    temp = h5py.File(i,'r')
    total+=temp['data'].attrs['total']
datagrp.attrs['total']=total

count=0
for current_path in paths:
    current_dataset = h5py.File(current_path)
    for demo in current_dataset['data']:
        demogrp = datagrp.create_group('demo_'+str(count))
        for k in current_dataset['data'][demo].attrs.keys():
            demogrp.attrs[k] = current_dataset['data'][demo].attrs[k]
        demogrp.attrs['og_demo_id']=demo
        demogrp.attrs['dataset_path']=current_path
        actionsdset = demogrp.create_dataset('actions', data = current_dataset['data'][demo]['actions'])
        rewardsdset = demogrp.create_dataset('rewards', data = current_dataset['data'][demo]['rewards'])
        statesdset = demogrp.create_dataset('states', data = current_dataset['data'][demo]['states'])
        donesdset = demogrp.create_dataset('dones', data = current_dataset['data'][demo]['dones'])
        obsgrp = demogrp.create_group('obs') 
        for grp_name in current_dataset['data'][demo]['obs']:
            dset = obsgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['obs'][grp_name])
        actiondictgrp = demogrp.create_group('action_dict') 
        for grp_name in current_dataset['data'][demo]['action_dict']:
            dset = actiondictgrp.create_dataset(grp_name, data = current_dataset['data'][demo]['action_dict'][grp_name])
        print('demo done', count)
        count += 1

    print('dataset done', current_path)
f.close()

demo done 0
demo done 1
demo done 2
demo done 3
demo done 4
demo done 5
demo done 6
demo done 7
demo done 8
demo done 9
demo done 10
demo done 11
demo done 12
demo done 13
demo done 14
demo done 15
demo done 16
demo done 17
demo done 18
demo done 19
demo done 20
demo done 21
demo done 22
demo done 23
demo done 24
demo done 25
demo done 26
demo done 27
demo done 28
demo done 29
demo done 30
demo done 31
demo done 32
demo done 33
demo done 34
demo done 35
demo done 36
demo done 37
demo done 38
demo done 39
demo done 40
demo done 41
demo done 42
demo done 43
demo done 44
demo done 45
demo done 46
demo done 47
demo done 48
demo done 49
demo done 50
demo done 51
demo done 52
dataset done /proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images.hdf5
demo done 53
demo done 54
demo done 55
demo done 56
demo done 57
demo done 58
demo done 59
demo done 60
demo done 61
demo done 62
demo done 63
demo done 64
d

In [140]:
'''TEST THE NEW DATASET'''

data = h5py.File(savepath, 'r')
print(data['data'])
video_path = base_path+'temp1.mp4'
video_writer = imageio.get_writer(video_path, fps=20)
idx = 6040
demo = f'demo_{idx}'
print(data['data'][demo]['object'])
print(data['data'][demo]['obs']['agentview_image'].shape)
print(data['data'][demo]['success'][:])
for b in  data['data'][demo]['obs']['agentview_image']:
    img = Image.fromarray((b).astype(np.uint8))
    d = ImageDraw.Draw(img)
    d.text( (2,2), str(idx), fill=255)
    
    
    #-- back to array
    b = np.asarray(img)

    video_writer.append_data(b)
    idx+=1
video_writer.close()
Video(video_path, embed=True)
data.close()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (84, 84) to (96, 96) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


<HDF5 group "/data" (6048 members)>
<HDF5 dataset "object": shape (), type "|O">
(100, 84, 84, 3)
[0.]


[swscaler @ 0x569b080] Warning: data is not aligned! This can lead to a speed loss
